In [1]:
import polars as pl
import datetime as dt
from sklearn.model_selection import TimeSeriesSplit
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline


EDA

Annual, monthly, weekly sales analysis

In [3]:
# 1. Load Raw Data
train_raw = pd.read_csv('store-sales-time-series-forecasting/train.csv')
test_raw = pd.read_csv('store-sales-time-series-forecasting/test.csv') # The Kaggle submission set

oil_raw = pd.read_csv('store-sales-time-series-forecasting/oil.csv')
stores_raw = pd.read_csv('store-sales-time-series-forecasting/stores.csv')
holidays_raw = pd.read_csv('store-sales-time-series-forecasting/holidays_events.csv')


In [4]:
# analysis of holidays

unique_holiday_type = holidays_raw['type'].unique()
print("Unique Holiday Types")
print(unique_holiday_type)

unique_holiday_locale = holidays_raw['locale'].unique()
print("Unique Holiday locale")
print(unique_holiday_locale)


Unique Holiday Types
<StringArray>
['Holiday', 'Transfer', 'Additional', 'Bridge', 'Work Day', 'Event']
Length: 6, dtype: str
Unique Holiday locale
<StringArray>
['Local', 'Regional', 'National']
Length: 3, dtype: str


In [19]:
# create custom preprocessor as standard scikit-learn pipelines don't natively handle relational
# database operations like pd.merge() or temporal operations like reindexing calendars.


class StoreSalesPreprocessor(BaseEstimator, TransformerMixin):
    """
    Custom Scikit-Learn Transformer for the Kaggle Store Sales Competition.
    Handles dates, oil interpolation, and relational table merges.
    """
    def __init__(self,
 oil_df, stores_df, holidays_df=None):
        # Pass the auxiliary dataframes when initializing the pipeline
        self.oil_df = oil_df.copy()
        self.stores_df = stores_df.copy()
        self.holidays_df = holidays_df.copy() if holidays_df is not None else None
        
    def fit(self, X, y=None):
        """
        The fit method learns parameters from the training data.
        We process the auxiliary tables here so it only happens once.
        """
        print("Fitting pipeline: Processing auxiliary tables...")
        
        # 1. Process Oil Data (The logic we discussed earlier)
        self.processed_oil_ = self._process_oil(self.oil_df)
        
        # 2. Process Stores Data (Any aggregations or cleaning on stores happens here)
        # For example, mapping 'city' or 'state' to broader regions if you wanted
        # self.processed_stores_ = self.stores_df.copy()
        self.processed_stores_ = self._process_stores(self.stores_df)

        self.processed_holidays_ = self._process_holidays(self.holidays_df)
        
        # 3. (Optional) Target Encoding Aggregations could go here
        # E.g., learning the historical mean sales per store from X and y
        
        return self

    def transform(self, X):
        """
        The transform method applies the merges to the Train or Test data.
        """
        print(f"Transforming data ({len(X)} rows)...")
        X_out = X.copy()
        
        # 1. Ensure date is a datetime object
        X_out['date'] = pd.to_datetime(X_out['date'])
        
        # 2. Merge Oil
        X_out = pd.merge(X_out, self.processed_oil_, on='date', how='left')
        
        # 3. Merge Stores
        X_out = pd.merge(X_out, self.processed_stores_, on='store_nbr', how='left')
        
        # 3. Merge Holidays
        X_out = pd.merge(X_out, self.processed_holidays_, on='date', how='left')
        
        # 4. Extract base calendar features (always good to have in the pipeline)
        X_out['day_of_week'] = X_out['date'].dt.dayofweek
        X_out['month'] = X_out['date'].dt.month
        X_out['year'] = X_out['date'].dt.year
        X_out['is_weekend'] = X_out['day_of_week'].isin([5, 6]).astype(int)
        
        return X_out

    def _process_oil(self, oil):
        """Helper method to encapsulate the oil interpolation logic"""
        oil['date'] = pd.to_datetime(oil['date'])
        
        # Create continuous calendar bounds based on the oil dataset
        calendar = pd.date_range(start=oil['date'].min(), end=oil['date'].max())
        
        # Reindex and Interpolate
        oil_continuous = oil.set_index('date').reindex(calendar).rename_axis('date').reset_index()
        
        # forward fill, then backward fill for the very first missing day (Jan 1, 2013)
        oil_continuous['dcoilwtico'] = oil_continuous['dcoilwtico'].ffill().bfill()
        
        # Rename column for clarity
        oil_continuous = oil_continuous.rename(columns={'dcoilwtico': 'oil_price'})
        return oil_continuous
    
    def _process_holidays(self, holidays):
        """Helper method to transform holidays so it can be incorporated with other tables"""
        holidays['date'] = pd.to_datetime(holidays['date'])

        holidays = holidays.rename(columns={'locale': 'holiday_location', 'locale_name': 'holiday_location_name', 'type': 'holiday_type', 'description': 'holiday_description',  'transferred': 'holiday_transferred'})

        return holidays
    
    def _process_stores(self, stores):
        """Helper method to transform stores so it can be incorporated with other tables"""

        stores = stores.rename(columns={'city': 'store_city', 'state': 'store_state', 'type': 'store_type', 'cluster': 'store_cluster'})

        return stores



In [20]:
# 2. Initialize your custom transformer
# Notice we pass the auxiliary tables into the initialization
feature_pipeline = Pipeline([
    ('preprocessor', StoreSalesPreprocessor(oil_df=oil_raw, stores_df=stores_raw, holidays_df=holidays_raw))
    # You can add StandardScalers or XGBoost models here later!
])

# 3. Fit and Transform the Training Data
# .fit() processes oil and stores. .transform() merges them into train.
X_train_processed = feature_pipeline.fit_transform(train_raw)

# 4. Transform the Test Data
# Reuses the exact same oil and store tables, ensuring 0 leakage and perfect consistency
X_test_processed = feature_pipeline.transform(test_raw)

# Look at the result
display(X_train_processed.head(100))


Fitting pipeline: Processing auxiliary tables...
Transforming data (3000888 rows)...
Transforming data (28512 rows)...


,id,date,store_nbr,family,sales,onpromotion,oil_price,store_city,store_state,store_type,store_cluster,holiday_type,holiday_location,holiday_location_name,holiday_description,holiday_transferred,day_of_week,month,year,is_weekend
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,93.14,Quito,Pichincha,D,13,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
1,1,2013-01-01,1,BABY CARE,0.0,0,93.14,Quito,Pichincha,D,13,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
2,2,2013-01-01,1,BEAUTY,0.0,0,93.14,Quito,Pichincha,D,13,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
3,3,2013-01-01,1,BEVERAGES,0.0,0,93.14,Quito,Pichincha,D,13,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
4,4,2013-01-01,1,BOOKS,0.0,0,93.14,Quito,Pichincha,D,13,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,2013-01-01,11,PREPARED FOODS,0.0,0,93.14,Cayambe,Pichincha,B,6,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
96,96,2013-01-01,11,PRODUCE,0.0,0,93.14,Cayambe,Pichincha,B,6,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
97,97,2013-01-01,11,SCHOOL AND OFFICE SUPPLIES,0.0,0,93.14,Cayambe,Pichincha,B,6,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0
98,98,2013-01-01,11,SEAFOOD,0.0,0,93.14,Cayambe,Pichincha,B,6,Holiday,National,Ecuador,Primer dia del ano,False,1,1,2013,0


In [21]:
display(X_train_processed[10000:10010])


,id,date,store_nbr,family,sales,onpromotion,oil_price,store_city,store_state,store_type,store_cluster,holiday_type,holiday_location,holiday_location_name,holiday_description,holiday_transferred,day_of_week,month,year,is_weekend
10000,10000,2013-01-06,4,BABY CARE,0.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10001,10001,2013-01-06,4,BEAUTY,5.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10002,10002,2013-01-06,4,BEVERAGES,1869.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10003,10003,2013-01-06,4,BOOKS,0.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10004,10004,2013-01-06,4,BREAD/BAKERY,518.348,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10005,10005,2013-01-06,4,CELEBRATION,0.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10006,10006,2013-01-06,4,CLEANING,1492.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10007,10007,2013-01-06,4,DAIRY,616.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10008,10008,2013-01-06,4,DELI,317.962,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1
10009,10009,2013-01-06,4,EGGS,325.000,0,93.12,Quito,Pichincha,D,9,NaN,NaN,NaN,NaN,NaN,6,1,2013,1


EDA braindump

Sales

Time Statistics

Yearly average sales
Monthly sales (year on year)
Weekly sales (year on year)
Daily sales (year on year)

Store Statistics

Sales by Store Type (over the years)
Sales by 

Sales by family (of item)

Autocorrelation, partial autocorrelation


## EDA

Our project has 7 datasets:

- holiday_events (all holidays, with typ, description, location and whether they were transferred or not)
- oil (oil price timeseries)
- stores (store identifier, location, type and cluster) - cluster is a grouping of similar stores
- transactions (number of transaction happening each day, per store - NB NOT transaction COST, just number of transactions) - also doesn't contain data for test days
- train (total sales, per store, item category, every day, on_promotion: gives the total number of items in a product family that were being promoted at a store at a given date.)

Overall main findings

Sales
- Average sales have steadily increased year on year. However, average number of transactions has remained steady over the years, indicating that inflation is behind increased average sales.
- 'Normal' year (2013 used as reference for this) consists of sales being fairly steady during year, with spike around november/december (christmas time)
- Earthquake which happened on April 16, 2016 caused unusual spike in sales for roughly 2 months (probably due to stockpiling)
- 2014 and 2015 had very unusual sales timeseries (inflation might have started in thes years) - 2014 sales very jagged (up and down constantly) - 2015 had unusual continuous increase in sales starting in May and continuing until end of year
- Sales by day of week - higher on U shape, skewed right (spikes on weekend, lowest in midweek)
- Sales by day of month - peaks at beginning of month - steadyish rest of month, with some dips towards end of month
- sales payday vs non payday - slightly higher on payday, but minimally
 

Stores
- store type A has significantly more sales than other store types (B, C, D, E)
- store cluster 5 has significantly more sales than other store types. store clusters 8, 11, 14, 17 also have significantly more sales than other store clusters
- stores in Quito make up 52% of total sales

Oil
- From Jan 2013 to June 2013, price was relatively stable
- Price had a hike from June 2013 to around October 2013 when it peaked
- By december 2013 prices were 'back to normal'
- Then prices steadily increased to July 2014
- Prices dropped dramatically from September 2014 to January 2015 - then increased slightly to June 2015, before dropping again (more than before) - minimum around January 2016 - 
- From January to June 2016 - steady increase
- June 2016 onwards - price is steady, but considerably lower than before in 2013

Holidays
- Holidays have some effect on sales (less than imagined) - Depends on type of holiday (Additional/Transfer/Bridge/Event/Work Day/Holiday/No Holiday), and whether holiday is national/regional/local
Products
- Top Product Families - groceries, then beverages, then produce



Note on Autocorrelation and Partial Autocorrelation

### 1. Does `nlags` affect the actual calculated values, or just the range?
**It only affects the range (the x-axis).** 
If you set `nlags=30` versus `nlags=40`, the calculated autocorrelation value at lag $k=20$ will be **exactly the same**. 

`nlags` is simply a parameter that tells the underlying algorithm (like `statsmodels.tsa.stattools.acf` or `pacf`) when to stop computing. The calculation for the correlation at lag $k$ depends strictly on the data from time $t=1$ to $t=N$ and the specific lag $k$. It does not depend on the maximum lag you requested.

*(Note: There are some obscure, highly specific matrix inversion methods for PACF that solve a full system up to `nlags`, but standard algorithms like Durbin-Levinson compute the AR models recursively. Thus, the coefficient for lag 20 is locked in once the algorithm reaches step 20, regardless of whether it stops at 30 or 40).*

---

### 2. How is autocorrelation at each value in the graph calculated?
Autocorrelation is essentially the Pearson correlation of a time series with a "shifted" (lagged) version of itself. However, there is a crucial computational difference in time series analysis to ensure the resulting autocorrelation matrix is positive semi-definite.

Let your time series be $Y_t$ of length $N$, with a global sample mean $\bar{Y}$.
The **autocovariance** at lag $k$ (let's call it $C_k$) is calculated as:

$$ C_k = \frac{1}{N} \sum_{t=1}^{N-k} (Y_t - \bar{Y})(Y_{t+k} - \bar{Y}) $$

The **autocorrelation** at lag $k$ ($r_k$) is simply normalized by the global variance $C_0$:

$$ r_k = \frac{C_k}{C_0} $$

**The Math Nuance:** Notice that we divide by $N$, not $N-k$, and we subtract the *global* mean $\bar{Y}$, not the local means of the overlapping $N-k$ subsets. If you just passed $Y_{1 \dots N-k}$ and $Y_{k+1 \dots N}$ to a standard Pearson correlation function like `scipy.stats.pearsonr`, you would get slightly different results. Time series libraries use the global mean and divide by $N$ to guarantee mathematically consistent behavior (e.g., preventing autocorrelations from exceeding 1 at high lags where $N-k$ is very small).

---

### 3. How is partial ACF (PACF) connected to ACF?
The ACF at lag $k$ measures the **total** correlation between $Y_t$ and $Y_{t-k}$. However, this total correlation includes "indirect" effects. 

For example, if today's sales are heavily correlated with yesterday's sales ($Y_t$ and $Y_{t-1}$), and yesterday's are correlated with the day before ($Y_{t-1}$ and $Y_{t-2}$), then $Y_t$ and $Y_{t-2}$ will have a high ACF value simply because $Y_{t-1}$ acts as a bridge. 

The **Partial Autocorrelation Function (PACF)** isolates the **direct** effect by removing the linear influence of the intermediate lags. 

Mathematically, the PACF at lag $k$ (denoted $\phi_{kk}$) is the $k$-th coefficient in an ordinary least squares (OLS) autoregressive model of order $k$, or $AR(k)$:

$$ Y_t = c + \phi_{k1} Y_{t-1} + \phi_{k2} Y_{t-2} + \dots + \mathbf{\phi_{kk}} Y_{t-k} + \epsilon_t $$

*   **At lag 1:** PACF is exactly the same as ACF.
*   **At lag 2:** PACF is the correlation between $Y_t$ and $Y_{t-2}$ *given* $Y_{t-1}$.
*   **At lag $k$:** It is the $k$-th coefficient $\phi_{kk}$, representing the unique variance in $Y_t$ explained *only* by $Y_{t-k}$.

If you have a purely Autoregressive process, say an $AR(1)$ where today only depends directly on yesterday, the **ACF** will trail off slowly toward zero (because of the echo effect), but the **PACF** will spike at lag 1 and drop instantly to 0 for all lags $\ge 2$.

---

### 4. What is the gray U-shaped curve that starts at 0 and extends rightwards?
That shaded area represents the **95% Confidence Interval for the Null Hypothesis** that the true autocorrelation is zero. 

Because of how you wrote your code (`lower_bound = confint[:, 0] - corr_array`), you subtracted the actual correlation values from the statsmodels confidence bounds. This elegantly centers the confidence interval exactly at $Y=0$ on your Plotly graph. 

*   If a blue dot (correlation) falls **inside** the gray area, it is statistically indistinguishable from 0 (white noise).
*   If a blue dot falls **outside** the gray area, it is considered a statistically significant lag.

**Why does it start at a width of 0 and open up into a U-shape (or funnel)?**
1.  **Lag 0:** At lag 0, a series is perfectly correlated with itself ($r_0 = 1$). The variance of this estimate is exactly 0. Thus, the confidence interval is literally $0 \pm 0$. That's why the funnel narrows to a single point on the far left.
2.  **Lag 1:** The standard error jumps to roughly $\pm 1.96 / \sqrt{N}$. 
3.  **Lags > 1 (The U-Shape in ACF):** By default, `statsmodels.tsa.stattools.acf` computes confidence intervals using **Bartlett’s formula**. Bartlett proved that if a series is a moving average process up to lag $q$, the variance of the sample autocorrelation at lags $k > q$ gets "inflated" by the squared autocorrelations of the previous lags:
    
    $$ \text{Var}(r_k) \approx \frac{1}{N} \left( 1 + 2 \sum_{i=1}^{k-1} r_i^2 \right) $$
    
    Because previous lags are being accumulated and squared, the standard error grows as the lag increases. This makes the bounds fan out, creating that U-shape.
4.  **Lags > 1 (PACF):** For PACF, Bartlett's formula does not apply. The standard error for PACF under the null hypothesis is constantly $1/\sqrt{N}$. If you look closely at your PACF graph, you should notice that after the point at lag 0, the gray band is actually a pair of flat, parallel lines, whereas the ACF band widens out continuously.

### How to Read and Interpret ACF and PACF Plots Side-by-Side

When analyzing time series data, reading the ACF and PACF graphs side-by-side is the foundation of the **Box-Jenkins methodology** for identifying ARIMA models. Here is a systematic guide to extracting insights from these plots.

#### Step 1: Check for Stationarity
Before you look for autoregressive (AR) or moving average (MA) patterns, you must check if the series is stationary.
*   **Non-Stationary:** If the ACF decays *extremely slowly* (e.g., a linear, gentle slope downwards across many lags) and the PACF has a massive spike at lag 1 (often close to 1.0), your data has a trend. You need to **difference** the series (e.g., $Y_t - Y_{t-1}$) before interpreting further.
*   **Stationary:** The ACF drops to zero relatively quickly (either cutting off sharply or decaying exponentially). 

#### Step 2: Check for Seasonality
Because you are looking at daily sales, seasonality is highly likely.
*   **Visual Cue:** Look for rhythmic spikes at specific intervals in the ACF. For daily data, if you see significant spikes at lags 7, 14, 21, etc., you have weekly seasonality. For monthly data, it would be lags 12, 24, 36.
*   **Action:** If seasonality is present, you usually apply a seasonal difference (e.g., $Y_t - Y_{t-7}$) or fit a Seasonal ARIMA (SARIMA) model.

#### Step 3: Identify the AR (p) and MA (q) Signatures
Once the data is stationary, you use the ACF and PACF side-by-side to guess the order of the Autoregressive $AR(p)$ and Moving Average $MA(q)$ processes. 

The general rule of thumb is: **The plot that "cuts off" (drops instantly to zero/into the confidence interval) gives you the order, while the other plot will "tail off" (decay slowly).**

| Data Generating Process | ACF Plot (Autocorrelation) | PACF Plot (Partial Autocorrelation) |
| :--- | :--- | :--- |
| **AR(p)** (Autoregressive) | **Tails off** (Decays exponentially or as a damped sine wave). | **Cuts off** sharply to zero after lag $p$. |
| **MA(q)** (Moving Average) | **Cuts off** sharply to zero after lag $q$. | **Tails off** (Decays exponentially or as a damped sine wave). |
| **ARMA(p, q)** (Mixed) | **Tails off** (Decays over time). | **Tails off** (Decays over time). |
| **White Noise** | No significant lags (all dots inside the gray confidence interval). | No significant lags (all dots inside the gray confidence interval). |

#### Step 4: Real-World Examples & Nuances
Textbook examples are perfectly clean, but real-world sales data is messy. Here is how to interpret common real-world edge cases:

*   **The "Echo" Effect:** If you see an $AR(1)$ process (PACF spikes at lag 1, then nothing), the ACF will naturally have a spike at lag 1, a smaller one at lag 2, a smaller one at lag 3. Do not add MA terms for those ACF spikes! They are just the mathematical "echo" of the AR(1) process.
*   **Over-differencing:** If your ACF has a significant negative spike at lag 1 (e.g., $-0.5$) and the PACF decays with alternating positive/negative spikes, you have likely over-differenced your data. You introduced artificial variance.
*   **When Both Tail Off:** If both plots decay slowly, you likely have a mixed ARMA process. The human eye is terrible at guessing exact $(p, q)$ orders when both tail off. In practice, use these plots to get a general neighborhood (e.g., $p \le 2, q \le 2$), and then let an algorithm minimize the **AIC** or **BIC** (Akaike/Bayesian Information Criterion) to select the exact model.


Stuff to investigate further in EDA

Holidays
- How often each holiday type occurs - if 'one off' then might be worth excluding type - bar chart with number of holidays, per type, over the years - also number of holidays per national/regional/local over the years

Generally
- We need to link geographical info contained within holidays and stores - someway to understand whether a regional/local holiday (holiday has locale_name) affects a given store (store has city and state)


Feature Engineering

For future - way to join holiday location with store location 


Calendar Features: Extract day_of_week, day_of_month, month, year, is_weekend, is_payday (e.g., 15th and last day of the month are huge for retail).


Lag Features: Sales from exactly 1 week ago ($t-7$), 2 weeks ago ($t-14$), etc.


Rolling Window Statistics: Moving averages, rolling standard deviations, rolling min/max over the last 7, 14, or 28 days.


Target Encoding: Average sales per store, average sales per product family.


Event Proximity: Days until the next holiday, days since the last holiday.
